# 🤼 Pro Kabaddi League - Exploratory Data Analysis & Visual Dashboard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adarshgaurav1951/Kabaddi-Matches-EDA/blob/main/notebooks/01_cleaning.ipynb)

A modular, step-by-step analytical workbook for exploring event-level Kabaddi match data, player profiling, franchise tactical matrices, and match momentum analytics.


### ⚙️ Step 1: Environment Setup & Data Ingestion

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Auto-detect Colab environment and clone repository data if needed
if 'google.colab' in sys.modules:
    if not os.path.exists('data'):
        print("Cloning repository files into Google Colab...")
        os.system('git clone https://github.com/adarshgaurav1951/Kabaddi-Matches-EDA.git _repo')
        os.system('mkdir -p data/cleaned data/raw')
        os.system('cp -r _repo/data/cleaned/* data/cleaned/ 2>/dev/null')
        os.system('cp -r _repo/data/raw/* data/raw/ 2>/dev/null')
        os.system('rm -rf _repo')
        print("Datasets successfully imported!")

# Directory paths
DATA_DIR = '../data/cleaned' if os.path.exists('../data/cleaned') else 'data/cleaned'
print(f"Data directory: {DATA_DIR}")


Data directory: ../data/cleaned


### 📂 Step 2: Load Cleaned Structured Datasets

In [2]:
# Ingest datasets
df_matches = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_matches.csv'))
df_teams = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_teams.csv'))
df_players = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_players.csv'))

df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')

print(f"Matches Dataset: {df_matches.shape[0]:,} rows | {df_matches.shape[1]} columns")
print(f"Teams Dataset:   {df_teams.shape[0]:,} rows | {df_teams.shape[1]} columns")
print(f"Players Dataset: {df_players.shape[0]:,} rows | {df_players.shape[1]} columns")


Matches Dataset: 684 rows | 18 columns
Teams Dataset:   1,368 rows | 26 columns
Players Dataset: 16,892 rows | 29 columns


### 📊 Step 3: Power BI Executive KPI Cards

In [3]:
tot_matches = len(df_matches)
tot_points = int(df_teams['score'].sum())
tot_raids = int(df_teams['total_raids'].sum())
tot_succ_raids = int(df_teams['successful_raids'].sum())
raid_strike = (tot_succ_raids / tot_raids * 100) if tot_raids else 0

tot_tackles = int(df_teams['total_tackles'].sum())
tot_succ_tackles = int(df_teams['successful_tackles'].sum())
tackle_eff = (tot_succ_tackles / tot_tackles * 100) if tot_tackles else 0

tot_all_outs = int(df_teams['all_outs_inflicted'].sum())
tot_supers = int(df_teams['super_raids'].sum())

display(HTML(f'''
<div style="display: flex; gap: 10px; margin: 10px 0; font-family: sans-serif;">
  <div style="flex: 1; background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 12px; text-align: center;">
    <div style="color: #8b949e; font-size: 11px; font-weight: 600; text-transform: uppercase;">Matches</div>
    <div style="color: #58a6ff; font-size: 22px; font-weight: bold;">{tot_matches:,}</div>
  </div>
  <div style="flex: 1; background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 12px; text-align: center;">
    <div style="color: #8b949e; font-size: 11px; font-weight: 600; text-transform: uppercase;">Total Points</div>
    <div style="color: #ffb703; font-size: 22px; font-weight: bold;">{tot_points:,}</div>
  </div>
  <div style="flex: 1; background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 12px; text-align: center;">
    <div style="color: #8b949e; font-size: 11px; font-weight: 600; text-transform: uppercase;">Raid Strike Rate</div>
    <div style="color: #00e5ff; font-size: 22px; font-weight: bold;">{raid_strike:.1f}%</div>
  </div>
  <div style="flex: 1; background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 12px; text-align: center;">
    <div style="color: #8b949e; font-size: 11px; font-weight: 600; text-transform: uppercase;">Tackle Success</div>
    <div style="color: #3fb950; font-size: 22px; font-weight: bold;">{tackle_eff:.1f}%</div>
  </div>
  <div style="flex: 1; background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 12px; text-align: center;">
    <div style="color: #8b949e; font-size: 11px; font-weight: 600; text-transform: uppercase;">All-Outs</div>
    <div style="color: #f78166; font-size: 22px; font-weight: bold;">{tot_all_outs:,}</div>
  </div>
  <div style="flex: 1; background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 12px; text-align: center;">
    <div style="color: #8b949e; font-size: 11px; font-weight: 600; text-transform: uppercase;">Super Raids</div>
    <div style="color: #a371f7; font-size: 22px; font-weight: bold;">{tot_supers:,}</div>
  </div>
</div>
'''))


### 🏆 Step 4: Franchise Scoring Averages

In [4]:
team_perf = df_teams.groupby('team_name').agg({
    'score': 'mean',
    'raid_points_total': 'mean',
    'tackle_points_total': 'mean',
    'all_outs_inflicted': 'sum',
    'match_id': 'count'
}).reset_index().rename(columns={'match_id': 'matches_played'})

team_perf = team_perf[team_perf['matches_played'] >= 10].sort_values(by='score', ascending=True)

fig_teams = px.bar(
    team_perf,
    x='score',
    y='team_name',
    orientation='h',
    color='score',
    color_continuous_scale="Blues",
    title="Average Points Scored Per Match by Franchise",
    labels={'score': 'Average Score', 'team_name': 'Franchise'},
    template="plotly_dark",
    height=450
)
fig_teams.update_layout(coloraxis_showscale=False)
fig_teams.show()


### ⚔️ Step 5: Tactical Matrix - Raid Power vs. Defensive Fortress

In [5]:
fig_matrix = px.scatter(
    team_perf,
    x='raid_points_total',
    y='tackle_points_total',
    text='team_name',
    size='all_outs_inflicted',
    color='score',
    color_continuous_scale="Turbo",
    title="Tactical Quadrants: Raid Power vs. Defensive Wall (Avg / Match)",
    labels={
        'raid_points_total': 'Avg Raid Points / Match',
        'tackle_points_total': 'Avg Tackle Points / Match',
        'score': 'Avg Match Score'
    },
    template="plotly_dark",
    height=500
)
fig_matrix.update_traces(textposition='top center')
fig_matrix.show()


### ⚡ Step 6: Top Raiders by Points & Points Per Raid (PPR)

In [6]:
raider_agg = df_players.groupby(['player_name', 'player_category']).agg({
    'player_raid_points_total': 'sum',
    'player_raids_total': 'sum',
    'player_raids_successful': 'sum'
}).reset_index()

raiders_q = raider_agg[raider_agg['player_raids_total'] >= 50].copy()
raiders_q['strike_rate'] = (raiders_q['player_raids_successful'] / raiders_q['player_raids_total']) * 100
raiders_q['ppr'] = raiders_q['player_raid_points_total'] / raiders_q['player_raids_total']

top12_raiders = raiders_q.sort_values(by='player_raid_points_total', ascending=False).head(12)

fig_raiders = px.bar(
    top12_raiders,
    x='player_raid_points_total',
    y='player_name',
    orientation='h',
    color='ppr',
    color_continuous_scale="YlOrRd",
    title="Top 12 Raiders by Career Points (Colored by Points Per Raid)",
    labels={'player_raid_points_total': 'Raid Points', 'player_name': 'Raider', 'ppr': 'PPR'},
    template="plotly_dark",
    height=450
)
fig_raiders.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_raiders.show()


### 🎯 Step 7: Physical Touch Points vs. Bonus Line Crossings

In [7]:
player_touch_bonus = df_players.groupby('player_name').agg({
    'player_raid_points_touch': 'sum',
    'player_raid_points_raid_bonus': 'sum',
    'player_raid_points_total': 'sum'
}).reset_index().sort_values(by='player_raid_points_total', ascending=False).head(10)

fig_split = go.Figure()
fig_split.add_trace(go.Bar(
    name='Touch Points',
    x=player_touch_bonus['player_name'],
    y=player_touch_bonus['player_raid_points_touch'],
    marker_color='#ff7b00'
))
fig_split.add_trace(go.Bar(
    name='Bonus Points',
    x=player_touch_bonus['player_name'],
    y=player_touch_bonus['player_raid_points_raid_bonus'],
    marker_color='#00e5ff'
))
fig_split.update_layout(
    barmode='stack',
    title="Raid Composition: Physical Touch vs. Bonus Points for Top Raiders",
    template="plotly_dark",
    height=400,
    xaxis_tickangle=-30
)
fig_split.show()


### 🛡️ Step 8: Top Defensive Anchors & Tackle Win Rate

In [8]:
def_agg = df_players.groupby(['player_name', 'player_category']).agg({
    'player_tackle_points_total': 'sum',
    'player_tackles_total': 'sum',
    'player_tackles_successful': 'sum'
}).reset_index()

def_q = def_agg[def_agg['player_tackles_total'] >= 30].copy()
def_q['conversion'] = (def_q['player_tackles_successful'] / def_q['player_tackles_total']) * 100

top12_def = def_q.sort_values(by='player_tackle_points_total', ascending=False).head(12)

fig_def = px.bar(
    top12_def,
    x='player_tackle_points_total',
    y='player_name',
    orientation='h',
    color='conversion',
    color_continuous_scale="Tealgrn",
    title="Top 12 Defenders by Tackle Points (Colored by Tackle Conversion %)",
    labels={'player_tackle_points_total': 'Tackle Points', 'player_name': 'Defender', 'conversion': 'Win Rate %'},
    template="plotly_dark",
    height=450
)
fig_def.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_def.show()


### ⏱️ Step 9: Match Victory Margin & Clutch Finishes (≤ 5 Points)

In [9]:
team_margins = df_teams.groupby('match_id')['score'].agg(
    lambda s: abs(s.iloc[0] - s.iloc[1]) if len(s) == 2 else 0
).reset_index()
team_margins.columns = ['match_id', 'margin']
team_margins['is_close_match'] = team_margins['margin'] <= 5

close_count = team_margins['is_close_match'].sum()
close_pct = (close_count / len(team_margins)) * 100

fig_margin = px.histogram(
    team_margins,
    x='margin',
    nbins=25,
    title=f"Match Victory Margin Distribution ({close_pct:.1f}% Clutches Decided by ≤ 5 Points)",
    color='is_close_match',
    color_discrete_map={True: '#ff7b00', False: '#58a6ff'},
    labels={'margin': 'Final Score Margin', 'is_close_match': 'Clutch Finish (≤5 pts)'},
    template="plotly_dark",
    height=420
)
fig_margin.show()


### 💥 Step 10: Score Escalation by All-Outs Inflicted

In [10]:
fig_allout = px.box(
    df_teams,
    x='all_outs_inflicted',
    y='score',
    color='all_outs_inflicted',
    title="Impact of All-Outs on Final Match Score",
    labels={'all_outs_inflicted': 'All-Outs Inflicted in Match', 'score': 'Team Final Score'},
    template="plotly_dark",
    height=420
)
fig_allout.update_layout(showlegend=False)
fig_allout.show()


### 🚀 Step 11: (Optional) Interactive Power BI Drag-and-Drop Builder in Colab

In [11]:
# Run this cell if you want a full Power BI / Tableau visual editor inside Colab!
try:
    import pygwalker as pyg
    pyg.walk(df_teams)
except ImportError:
    print("To enable interactive Power BI drag-and-drop in Colab, run:")
    print("!pip install pygwalker -q")


To enable interactive Power BI drag-and-drop in Colab, run:
!pip install pygwalker -q
